In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [58]:
import os
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.utils import save_image
from PIL import Image

In [59]:
ROOT_DIR = "/content/drive/MyDrive/DDI"
IMG_DIR = os.path.join(ROOT_DIR, "images")
CSV_PATH = os.path.join(ROOT_DIR, "ddi_metadata.csv")
SAVE_DIR = os.path.join(ROOT_DIR, "gan_checkpoints")
os.makedirs(SAVE_DIR, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

IMAGE_SIZE = 128
Z_DIM = 100
BATCH_SIZE = 8
EPOCHS = 300
LR = 2e-4


In [69]:
df = pd.read_csv(CSV_PATH)

IMAGE_COL = "DDI_file"
SKIN_COL  = "skin_tone"

df = df[df[SKIN_COL] < 34].reset_index(drop=True)

print("Filtered images:", len(df))
print("Max skin tone:", df[SKIN_COL].max())

assert len(df) > 0, "No images after filtering!"

Filtered images: 208
Max skin tone: 12


In [61]:
class DDIBlackSkinDataset(Dataset):
    def __init__(self, dataframe, img_dir, transform=None):
        self.df = dataframe
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_name = self.df.loc[idx, IMAGE_COL]

        if not img_name.lower().endswith((".jpg", ".png", ".jpeg")):
            img_name += ".jpg"

        img_path = os.path.join(self.img_dir, img_name)
        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image


In [62]:
transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(0.5),
    transforms.RandomRotation(10),
    transforms.ColorJitter(0.1, 0.1, 0.1, 0.05),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

In [63]:
dataset = DDIBlackSkinDataset(df, IMG_DIR, transform)

dataloader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)
print("Dataset ready:", len(dataset))

Dataset ready: 208


In [77]:
Z_DIM = 100
IMG_CHANNELS = 3
FEATURES_G = 64
FEATURES_D = 64

BATCH_SIZE = 4
EPOCHS = 200
LR = 2e-4
SAVE_DIR = "/content/drive/MyDrive/DDI/updated_gan_checkpoints"
os.makedirs(SAVE_DIR, exist_ok=True)


In [64]:
class Generator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.ConvTranspose2d(Z_DIM, 1024, 4, 1, 0),
            nn.BatchNorm2d(1024),
            nn.ReLU(True),

            nn.ConvTranspose2d(1024, 512, 4, 2, 1),
            nn.BatchNorm2d(512),
            nn.ReLU(True),

            nn.ConvTranspose2d(512, 256, 4, 2, 1),
            nn.BatchNorm2d(256),
            nn.ReLU(True),

            nn.ConvTranspose2d(256, 128, 4, 2, 1),
            nn.BatchNorm2d(128),
            nn.ReLU(True),

            nn.ConvTranspose2d(128, 64, 4, 2, 1),
            nn.BatchNorm2d(64),
            nn.ReLU(True),

            nn.ConvTranspose2d(64, 3, 4, 2, 1),
            nn.Tanh()
        )

    def forward(self, z):
        return self.net(z)

In [65]:
class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.utils.spectral_norm(nn.Conv2d(3, 64, 4, 2, 1)),
            nn.LeakyReLU(0.2),

            nn.utils.spectral_norm(nn.Conv2d(64, 128, 4, 2, 1)),
            nn.LeakyReLU(0.2),

            nn.utils.spectral_norm(nn.Conv2d(128, 256, 4, 2, 1)),
            nn.LeakyReLU(0.2),

            nn.utils.spectral_norm(nn.Conv2d(256, 512, 4, 2, 1)),
            nn.LeakyReLU(0.2),

            nn.utils.spectral_norm(nn.Conv2d(512, 1024, 4, 2, 1)),
            nn.LeakyReLU(0.2),

            nn.Conv2d(1024, 1, 4, 1, 0)
        )

    def forward(self, x):
        return self.net(x).view(-1)


In [66]:
G = Generator().to(DEVICE)
D = Discriminator().to(DEVICE)

opt_G = optim.Adam(G.parameters(), lr=LR, betas=(0.5, 0.999))
opt_D = optim.Adam(D.parameters(), lr=LR, betas=(0.5, 0.999))

criterion = nn.BCEWithLogitsLoss()
fixed_noise = torch.randn(16, Z_DIM, 1, 1).to(DEVICE)

In [78]:
best_g_loss = float("inf")
patience = 30
no_improve_epochs = 0

for epoch in range(1, EPOCHS + 1):
    epoch_g_loss = 0.0
    epoch_d_loss = 0.0
    num_batches = 0

    for real in dataloader:
        real = real.to(DEVICE)
        real = real + 0.05 * torch.randn_like(real)
        batch_size = real.size(0)

        noise = torch.randn(batch_size, Z_DIM, 1, 1).to(DEVICE)
        fake = G(noise)

        real_labels = torch.full((batch_size,), 0.9, device=DEVICE)
        fake_labels = torch.zeros(batch_size, device=DEVICE)

        loss_D = (
            criterion(D(real), real_labels) +
            criterion(D(fake.detach()), fake_labels)
        )

        opt_D.zero_grad()
        loss_D.backward()
        opt_D.step()

        loss_G = criterion(D(fake), real_labels)

        opt_G.zero_grad()
        loss_G.backward()
        opt_G.step()

        epoch_g_loss += loss_G.item()
        epoch_d_loss += loss_D.item()
        num_batches += 1

    avg_g_loss = epoch_g_loss / num_batches
    avg_d_loss = epoch_d_loss / num_batches

    print(
        f"Epoch [{epoch}/{EPOCHS}] | "
        f"D: {avg_d_loss:.4f} | G: {avg_g_loss:.4f}"
    )

    if avg_g_loss < best_g_loss:
        best_g_loss = avg_g_loss
        no_improve_epochs = 0

        torch.save(G.state_dict(), f"{SAVE_DIR}/G_best.pth")
        print(f"✅ Best Generator saved (G loss: {best_g_loss:.4f})")

        with torch.no_grad():
            samples = G(fixed_noise)
            save_image(
                samples,
                f"{SAVE_DIR}/samples_best.png",
                normalize=True
            )
    else:
        no_improve_epochs += 1

    if no_improve_epochs >= patience:
        print(
            f"Early stopping triggered "
            f"(no G improvement for {patience} epochs)"
        )
        break


Epoch [1/200] | D: 0.8177 | G: 2.2403
✅ Best Generator saved (G loss: 2.2403)
Epoch [2/200] | D: 0.8385 | G: 2.4984
Epoch [3/200] | D: 0.8491 | G: 2.1152
✅ Best Generator saved (G loss: 2.1152)
Epoch [4/200] | D: 0.8365 | G: 2.6985
Epoch [5/200] | D: 0.8543 | G: 2.2946
Epoch [6/200] | D: 0.8610 | G: 2.3023
Epoch [7/200] | D: 0.8644 | G: 2.2807
Epoch [8/200] | D: 0.8868 | G: 2.1737
Epoch [9/200] | D: 0.8766 | G: 2.2535
Epoch [10/200] | D: 0.8846 | G: 2.2515
Epoch [11/200] | D: 0.8850 | G: 2.2592
Epoch [12/200] | D: 0.8530 | G: 2.4053
Epoch [13/200] | D: 0.7637 | G: 2.4996
Epoch [14/200] | D: 0.8398 | G: 2.5069
Epoch [15/200] | D: 0.9783 | G: 2.2081
Epoch [16/200] | D: 0.8599 | G: 1.9916
✅ Best Generator saved (G loss: 1.9916)
Epoch [17/200] | D: 0.8474 | G: 2.1514
Epoch [18/200] | D: 0.9011 | G: 2.2792
Epoch [19/200] | D: 0.8500 | G: 2.2287
Epoch [20/200] | D: 0.8895 | G: 2.1776
Epoch [21/200] | D: 0.8516 | G: 2.2899
Epoch [22/200] | D: 0.9197 | G: 2.3257
Epoch [23/200] | D: 0.8427 | G:

In [73]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

Z_DIM = 100
IMG_SIZE = 64
IMG_CHANNELS = 3

MODEL_PATH = "/content/drive/MyDrive/DDI/gan_checkpoints/G_epoch_25.pth"
OUTPUT_DIR = "/content/drive/MyDrive/DDI/generated_images"

os.makedirs(OUTPUT_DIR, exist_ok=True)


In [74]:
noise = torch.randn(1, Z_DIM, 1, 1).to(DEVICE)

with torch.no_grad():
    fake_image = G(noise)

save_image(
    fake_image,
    f"{OUTPUT_DIR}/generated_single.png",
    normalize=True
)

print("Single image generated")


Single image generated
